# Coaching360 demo queries

One section per demo question: the question, who asks it, the expected answer, and the SQL that produces it.

- Database: `coaching360_demo.sqlite` in this folder, opened **read-only**.
- Run the setup cell first, then any question cell.
- Results show as pandas tables if pandas is installed; otherwise as plain text.

In [1]:
# Optional: install pandas for nicer tables, then restart the kernel
# %pip install pandas

In [2]:
import sqlite3
from pathlib import Path

DB = Path("coaching360_demo.sqlite")
assert DB.exists(), f"{DB.resolve()} not found; open this notebook from the coaching360-demo-dataset folder"
conn = sqlite3.connect(f"file:{DB}?mode=ro", uri=True)

try:
    import pandas as pd
    pd.set_option("display.max_colwidth", 200)
except ImportError:
    pd = None


def run(sql):
    """Run a query and display the result as a table."""
    if pd is not None:
        return pd.read_sql_query(sql, conn)
    cur = conn.execute(sql)
    cols = [c[0] for c in cur.description]
    rows = [["" if v is None else str(v) for v in r] for r in cur.fetchall()]
    widths = [min(60, max(len(c), *(len(r[i]) for r in rows))) if rows else len(c) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v[:w].ljust(w) for v, w in zip(vals, widths))
    print(line(cols))
    print("-+-".join("-" * w for w in widths))
    for r in rows:
        print(line(r))
    print(f"({len(rows)} rows)")

## Act 1 · Scorecards

### Q1. Show me my scorecard for September 2026.

*As Rowan Jessup* · `/coaching360 Rowan Jessup`

**Expected answer:** LPE 0.29 (goal 1.00), 6 loans; DACC loans 2 (goal 2); 30–89 day DLQ 4.09% (goal 3.40%); first lien 32.6%; cash advance ~$56.9K (state avg ~$67.1K); outbound calls 1,102 (goal 1,200); contacts 252 (goal 300).

In [3]:
run("""
SELECT '2026-09-01' AS KpiMonth, b.BranchName, p.TotalLoans, p.LPE, 1.0 AS LpeGoal, p.DACC_Loans, 2 AS DaccLoansGoal, p.DLQCurrentMonthPerc_Actual AS DlqCurrentMonth, p.DLQCurrentMonthPerc_Goal AS DlqCurrentMonthGoal, p.DLQ3060DayPerc_Actual AS Dlq30To89Day, p.DLQ3089DayPerc_Goal AS Dlq30To89DayGoal, p.FirstLienPercentage, p.Employee_Cash_Advance AS CashAdvance, p.Average_State_Cash_Advance AS StateAverageCashAdvance, c.OutboundCalls, gc.TargetValue AS OutboundCallsGoal, c.OutboundContacts, gt.TargetValue AS OutboundContactsGoal
FROM org_hierarchy__EDA_Ref_T_Ref_User u
  JOIN branchkpi_sources__BranchKPI_Monthly_EmployeePerformance p ON p.TellerID=u.TellerID AND p.Year=2026 AND p.Month=9
  LEFT JOIN org_hierarchy__EDA_Ref_T_Ref_Branch b ON b.BranchNumber=p.Branch
  LEFT JOIN (SELECT TellerID, SUM(OutboundCalls) AS OutboundCalls, SUM(OutboundContacts) AS OutboundContacts
FROM branchkpi_sources__Reports_KPIDashboard_CallData
WHERE call_date>='2026-09-01' AND call_date<'2026-10-01'
GROUP BY TellerID) c ON c.TellerID=u.TellerID
  LEFT JOIN branchkpi_sources__Reports_KPIDashboard_EmployeeGoals gc ON gc.TellerID=u.TellerID AND gc.GoalID=1
  LEFT JOIN branchkpi_sources__Reports_KPIDashboard_EmployeeGoals gt ON gt.TellerID=u.TellerID AND gt.GoalID=2
WHERE u.UserDWID=500079
""")

KpiMonth   | BranchName  | TotalLoans | LPE    | LpeGoal | DACC_Loans | DaccLoansGoal | DlqCurrentMonth | DlqCurrentMonthGoal | Dlq30To89Day | Dlq30To89DayGoal | FirstLienPercentage | CashAdvance | StateAverageCashAdvance | OutboundCalls | OutboundCallsGoal | OutboundContacts | OutboundContactsGoal
-----------+-------------+------------+--------+---------+------------+---------------+-----------------+---------------------+--------------+------------------+---------------------+-------------+-------------------------+---------------+-------------------+------------------+---------------------
2026-09-01 | Laurel Fork | 6          | 0.2857 | 1.0     | 2          | 2             | 0.083406        | 0.08                | 0.040913     | 0.034            | 32.56               | 56857.77    | 67145.01                | 1102          | 1200.0            | 252              | 300.0               
(1 rows)


### Q2. How does my LPE compare to my goal and to last month?

*As Rowan Jessup* · `/coaching360 Rowan Jessup`

**Expected answer:** 0.29 vs goal 1.00, down from 0.35 in August. YTD average 0.52.

In [4]:
run("""
SELECT MAX(CASE WHEN p.Month=9 THEN p.LPE END) AS SeptemberLPE, 1.0 AS LpeGoal, MAX(CASE WHEN p.Month=8 THEN p.LPE END) AS AugustLPE, AVG(p.LPE) AS YearToDateAvgLPE
FROM org_hierarchy__EDA_Ref_T_Ref_User u
  JOIN branchkpi_sources__BranchKPI_Monthly_EmployeePerformance p ON p.TellerID=u.TellerID
WHERE u.UserDWID=500079 AND p.Year=2026 AND p.Month BETWEEN 1 AND 9
""")

SeptemberLPE | LpeGoal | AugustLPE | YearToDateAvgLPE  
-------------+---------+-----------+-------------------
0.2857       | 1.0     | 0.35      | 0.5151333333333333
(1 rows)


### Q3. Which of my September metrics met goal and which missed?

*As Rowan Jessup* · `/coaching360 Rowan Jessup`

**Expected answer:** On goal: DACC loans (2 of 2). Below: LPE, 30–89 DLQ (4.09% vs 3.40%, improved from 4.67%), cash advance, calls (1,102/1,200), contacts (252/300).

In [ ]:
run("""
SELECT CASE WHEN p.LPE>=1.0 THEN 'Met' ELSE 'Missed' END AS LPE, CASE WHEN p.DACC_Loans>=2 THEN 'Met' ELSE 'Missed' END AS DACC, CASE WHEN p.DLQCurrentMonthPerc_Actual<=p.DLQCurrentMonthPerc_Goal THEN 'Met' ELSE 'Missed' END AS DlqCurrent, CASE WHEN p.DLQ3060DayPerc_Actual<=p.DLQ3089DayPerc_Goal THEN 'Met' ELSE 'Missed' END AS Dlq30To89, CASE WHEN p.Employee_Cash_Advance>=p.Average_State_Cash_Advance THEN 'Met' ELSE 'Missed' END AS CashAdvance, CASE WHEN c.OutboundCalls>=gc.TargetValue THEN 'Met' ELSE 'Missed' END AS Calls, CASE WHEN c.OutboundContacts>=gt.TargetValue THEN 'Met' ELSE 'Missed' END AS Contacts
FROM org_hierarchy__EDA_Ref_T_Ref_User u
  JOIN branchkpi_sources__BranchKPI_Monthly_EmployeePerformance p ON p.TellerID=u.TellerID AND p.Year=2026 AND p.Month=9
  LEFT JOIN (SELECT TellerID, SUM(OutboundCalls) AS OutboundCalls, SUM(OutboundContacts) AS OutboundContacts
FROM branchkpi_sources__Reports_KPIDashboard_CallData
WHERE call_date>='2026-09-01' AND call_date<'2026-10-01'
GROUP BY TellerID) c ON c.TellerID=u.TellerID
  LEFT JOIN branchkpi_sources__Reports_KPIDashboard_EmployeeGoals gc ON gc.TellerID=u.TellerID AND gc.GoalID=1
  LEFT JOIN branchkpi_sources__Reports_KPIDashboard_EmployeeGoals gt ON gt.TellerID=u.TellerID AND gt.GoalID=2
WHERE u.UserDWID=500079
""")

LPE    | DACC | DlqCurrent | Dlq30To89 | CashAdvance | Calls  | Contacts
-------+------+------------+-----------+-------------+--------+---------
Missed | Met  | Missed     | Missed    | Missed      | Missed | Missed  
(1 rows)


### Q4. Show me my team's scorecard at Laurel Fork for September.

*As Riley Quimby* · `/coaching360 Riley Quimby`

**Expected answer:** Selena Driscoll 0.95 LPE, 4 DACC, 2.98% DLQ, 54.6% first lien · Avery Northcott 0.86, 1, 4.57%, 43.2% · Riley Quimby 0.48, 0, 5.11%, 27.5% · Rowan Jessup 0.29, 2, 4.09%, 32.6%. Branch: 0.64 LPE, 7 DACC, 4.19% DLQ, 42.2% first lien.

In [ ]:
run("""
SELECT u.Name AS EmployeeName, u.UserTitle, p.TotalLoans, p.LPE, p.DACC_Loans, p.DLQCurrentMonthPerc_Actual AS DlqCurrentMonth, p.DLQ3060DayPerc_Actual AS Dlq30To89Day, p.FirstLienPercentage, c.OutboundCalls, c.OutboundContacts
FROM branchkpi_sources__BranchKPI_Monthly_EmployeePerformance p
  JOIN org_hierarchy__EDA_Ref_T_Ref_User u ON u.TellerID=p.TellerID
  JOIN org_hierarchy__EDA_Ref_T_Ref_Branch b ON b.BranchNumber=p.Branch
  LEFT JOIN (SELECT TellerID, SUM(OutboundCalls) AS OutboundCalls, SUM(OutboundContacts) AS OutboundContacts
FROM branchkpi_sources__Reports_KPIDashboard_CallData
WHERE call_date>='2026-09-01' AND call_date<'2026-10-01'
GROUP BY TellerID) c ON c.TellerID=p.TellerID
WHERE b.BranchName='Laurel Fork' AND p.Year=2026 AND p.Month=9
ORDER BY p.LPE DESC
""")

EmployeeName    | UserTitle                | TotalLoans | LPE    | DACC_Loans | DlqCurrentMonth | Dlq30To89Day | FirstLienPercentage | OutboundCalls | OutboundContacts
----------------+--------------------------+------------+--------+------------+-----------------+--------------+---------------------+---------------+-----------------
Selena Driscoll | Assistant Branch Manager | 20         | 0.9524 | 4          | 0.071149        | 0.029812     | 54.58               | 1805          | 416             
Avery Northcott | Account Representative   | 18         | 0.8571 | 1          | 0.068515        | 0.045748     | 43.17               | 1443          | 329             
Riley Quimby    | Branch Manager           | 10         | 0.4762 | 0          | 0.105724        | 0.051113     | 27.51               | 431           | 91              
Rowan Jessup    | Account Representative   | 6          | 0.2857 | 2          | 0.083406        | 0.040913     | 32.56               | 1102          | 252      

In [ ]:
run("""
SELECT Direct_Loans_MTD_LPE, Branch_DACC_Loans, DLQ3060DayPerc_Actual*100, FirstLienPercentage
FROM branchkpi_sources__BranchKPI_Monthly_BranchPerformance
WHERE Branch=8011 AND Year=2026 AND Month=9
""")

Direct_Loans_MTD_LPE | Branch_DACC_Loans | DLQ3060DayPerc_Actual*100 | FirstLienPercentage
---------------------+-------------------+---------------------------+--------------------
0.6429               | 7                 | 4.1896                    | 42.19              
(1 rows)


### Q5. Who is the top performer at Laurel Fork, and who needs the most help?

*As Riley Quimby* · `/coaching360 Riley Quimby`

**Expected answer:** Across Riley's scope incl. relief branch: Andre Valdez (1.24 LPE, Shenandoah Mills). At Laurel Fork: Selena Driscoll (0.95 LPE, only one under the DLQ goal, 4 DACC, 54.6% first lien). Needs help: Rowan Jessup (0.29 LPE, down from 0.48 in June despite repeated coaching).

In [ ]:
run("""
SELECT u.Name AS EmployeeName, p.LPE, p.DACC_Loans, p.DLQ3060DayPerc_Actual AS Dlq30To89Day, p.FirstLienPercentage, RANK() OVER (ORDER BY p.LPE DESC) AS LpeRank
FROM branchkpi_sources__BranchKPI_Monthly_EmployeePerformance p
  JOIN org_hierarchy__EDA_Ref_T_Ref_User u ON u.TellerID=p.TellerID
  JOIN org_hierarchy__EDA_Ref_T_Ref_Branch b ON b.BranchNumber=p.Branch
WHERE b.BranchName='Laurel Fork' AND p.Year=2026 AND p.Month=9
ORDER BY p.LPE DESC
""")

EmployeeName    | LPE    | DACC_Loans | Dlq30To89Day | FirstLienPercentage | LpeRank
----------------+--------+------------+--------------+---------------------+--------
Selena Driscoll | 0.9524 | 4          | 0.029812     | 54.58               | 1      
Avery Northcott | 0.8571 | 1          | 0.045748     | 43.17               | 2      
Riley Quimby    | 0.4762 | 0          | 0.051113     | 27.51               | 3      
Rowan Jessup    | 0.2857 | 2          | 0.040913     | 32.56               | 4      
(4 rows)


### Q6. Which employees are over the 30–89 day delinquency goal?

*As Riley Quimby* · `/coaching360 Riley Quimby`

**Expected answer:** Goal 3.40%: Reese Delacroix (Shenandoah Mills, via relief coverage) 5.24%, Riley Quimby 5.11%, Avery Northcott 4.57%, Rowan Jessup 4.09%.

In [ ]:
run("""
SELECT u.Name AS EmployeeName, b.BranchName, p.DLQ3060DayPerc_Actual AS Dlq30To89Day, p.DLQ3089DayPerc_Goal AS Dlq30To89DayGoal
FROM branchkpi_sources__BranchKPI_Monthly_EmployeePerformance p
  JOIN org_hierarchy__EDA_Ref_T_Ref_User u ON u.TellerID=p.TellerID
  JOIN org_hierarchy__EDA_Ref_T_Ref_Branch b ON b.BranchNumber=p.Branch
WHERE b.BranchName IN ('Laurel Fork','Shenandoah Mills') AND p.Year=2026 AND p.Month=9 AND p.DLQ3060DayPerc_Actual>p.DLQ3089DayPerc_Goal
ORDER BY p.DLQ3060DayPerc_Actual DESC
""")

EmployeeName    | BranchName       | Dlq30To89Day | Dlq30To89DayGoal
----------------+------------------+--------------+-----------------
Reese Delacroix | Shenandoah Mills | 0.052402     | 0.034           
Riley Quimby    | Laurel Fork      | 0.051113     | 0.034           
Avery Northcott | Laurel Fork      | 0.045748     | 0.034           
Rowan Jessup    | Laurel Fork      | 0.040913     | 0.034           
(4 rows)


## Act 2 · Success Plan

### Q7. What's in my Success Plan for October?

*As Rowan Jessup* · `/coaching360 Rowan Jessup`

**Expected answer:** Active, not yet approved. Metrics: LPE, Contacts per Worked Hour, First Lien Rate. Competency: Drive Results. Goal: "Build consistency in LPE through daily routines."

In [ ]:
run("""
SELECT GoalMonth, Status, ApprovedUtc, MetricFocusAreas, CompetencyFocusAreas, GoalStatement, ActionPlan, SuccessMeasures
FROM coaching360_app_workflow__dbo_IndividualDevelopmentGoals
WHERE EmployeeId=500079 AND date(GoalMonth, 'start of month')='2026-10-01'
""")

GoalMonth               | Status | ApprovedUtc | MetricFocusAreas                                     | CompetencyFocusAreas | GoalStatement                                    | ActionPlan                                                   | SuccessMeasures                                             
------------------------+--------+-------------+------------------------------------------------------+----------------------+--------------------------------------------------+--------------------------------------------------------------+-------------------------------------------------------------
2026-10-01 00:00:00.000 | Active |             | ["LPE","Contacts per Worked Hour","First Lien Rate"] | ["Drive Results"]    | Build consistency in LPE through daily routines. | Ask the vehicle ownership question on every application; pre | LPE at or above 1.00, Contacts per Worked Hour at or above 9
(1 rows)


### Q8. What did I focus on in September, and did I hit it?

*As Rowan Jessup* · `/coaching360 Rowan Jessup`

**Expected answer:** Plan is locked. LPE and DACC Loans; Communicates Effectively. DACC hit (2 vs 2); LPE missed (0.29 vs 1.00).

In [ ]:
run("""
SELECT g.GoalMonth, g.MetricFocusAreas, g.CompetencyFocusAreas, g.ApprovedUtc, g.LockedUtc, p.LPE, 1.0 AS LpeGoal, p.DACC_Loans, 2 AS DaccLoansGoal
FROM coaching360_app_workflow__dbo_IndividualDevelopmentGoals g
  JOIN org_hierarchy__EDA_Ref_T_Ref_User u ON u.UserDWID=g.EmployeeId
  JOIN branchkpi_sources__BranchKPI_Monthly_EmployeePerformance p ON p.TellerID=u.TellerID AND p.Year=2026 AND p.Month=9
WHERE g.EmployeeId=500079 AND date(g.GoalMonth, 'start of month')='2026-09-01'
""")

GoalMonth               | MetricFocusAreas     | CompetencyFocusAreas         | ApprovedUtc             | LockedUtc               | LPE    | LpeGoal | DACC_Loans | DaccLoansGoal
------------------------+----------------------+------------------------------+-------------------------+-------------------------+--------+---------+------------+--------------
2026-09-01 00:00:00.000 | ["LPE","DACC Loans"] | ["Communicates Effectively"] | 2026-09-03 20:50:22.142 | 2026-10-01 04:03:18.000 | 0.2857 | 1.0     | 2          | 2            
(1 rows)


### Q9. Which Success Plans on my team still need my approval?

*As Riley Quimby* · `/coaching360 Riley Quimby`

**Expected answer:** Three Active: Rowan Jessup (Laurel Fork), Andre Valdez and Reese Delacroix (Shenandoah Mills, via relief coverage; Reese's due 10/31). Andre Everhart's is already Approved. Selena and Avery have no October plan.

In [ ]:
run("""
SELECT u.Name, b.BranchName, g.GoalMonth, g.Status, g.DueDate
FROM coaching360_app_workflow__dbo_IndividualDevelopmentGoals g
  JOIN org_hierarchy__EDA_Ref_T_Ref_User u ON u.UserDWID=g.EmployeeId
  JOIN org_hierarchy__EDA_Ref_T_Ref_Branch b ON b.BranchDWID=u.BranchDWID
WHERE b.BranchName IN ('Laurel Fork','Shenandoah Mills') AND g.EmployeeId<>500072 AND g.Status='Active' AND g.ApprovedUtc IS NULL
ORDER BY g.GoalMonth, b.BranchName, u.Name
""")

Name            | BranchName       | GoalMonth               | Status | DueDate                
----------------+------------------+-------------------------+--------+------------------------
Rowan Jessup    | Laurel Fork      | 2026-10-01 00:00:00.000 | Active |                        
Andre Valdez    | Shenandoah Mills | 2026-10-01 00:00:00.000 | Active |                        
Reese Delacroix | Shenandoah Mills | 2026-10-01 00:00:00.000 | Active | 2026-10-31 00:00:00.000
(3 rows)


## Act 3 · Coaching

### Q10. Show me Rowan Jessup's coaching history.

*As Riley Quimby* · `/coaching360 Riley Quimby`

**Expected answer:** 16 records this year, half on KPIs, LPE recurring. Latest: 09-14 accountability call on LPE; 09-01 Action Oriented; 08-07 cash advance; 07-30 branch visit LPE (pending ack); 07-10 KPI LPE; 06-29 Collaborates.

In [ ]:
run("""
SELECT r.CoachingDate, r.TriggerType, COALESCE(r.TopicCode, r.CompetencyCode) AS Topic, r.KpiFocusCode, r.ObservationSummary, r.Status
FROM coaching360_app_workflow__dbo_CoachingRecords r
  JOIN org_hierarchy__EDA_Ref_T_Ref_User u ON u.UserDWID=r.EmployeeId
WHERE u.Name='Rowan Jessup'
ORDER BY r.CoachingDate DESC
""")

CoachingDate            | TriggerType        | Topic               | KpiFocusCode | ObservationSummary                                           | Status                
------------------------+--------------------+---------------------+--------------+--------------------------------------------------------------+-----------------------
2026-09-14 00:00:00.000 | accountabilityCall | performance         | lpe          | Rowan finished August at 0.35 LPE; pipeline reviews show sev | Submitted             
2026-09-01 00:00:00.000 | competency         | development         |              | Coaching conversation on Action Oriented.                    | Submitted             
2026-08-07 00:00:00.000 | kpi                | performance         | cashAdvance  | Net cash advanced was below the state average for Rowan in J | Submitted             
2026-07-30 00:00:00.000 | branchVisit        | performance         | lpe          | Rowan finished June at 0.48 LPE; pipeline reviews show sever | Pen

### Q11. What did I coach Rowan on last, and what was the action plan?

*As Riley Quimby* · `/coaching360 Riley Quimby`

**Expected answer:** 09-14 accountability call on LPE; root cause process gap. Walked through the daily pipeline routine and role-played the closing ask. Action: 5 closed loans per week. Rowan finished September with 6 loans.

In [ ]:
run("""
SELECT r.CoachingDate, r.TriggerType, r.KpiFocusCode, r.CoachingProvided, r.RootCause, r.ActionRequired
FROM coaching360_app_workflow__dbo_CoachingRecords r
  JOIN org_hierarchy__EDA_Ref_T_Ref_User u ON u.UserDWID=r.EmployeeId
WHERE u.Name='Rowan Jessup'
ORDER BY r.CoachingDate DESC
LIMIT 1
""")

CoachingDate            | TriggerType        | KpiFocusCode | CoachingProvided                                             | RootCause  | ActionRequired                                           
------------------------+--------------------+--------------+--------------------------------------------------------------+------------+----------------------------------------------------------
2026-09-14 00:00:00.000 | accountabilityCall | lpe          | Walked through the daily pipeline routine: first-hour callba | processGap | Target 5 closed loans per week for the rest of the month.
(1 rows)


### Q12. Show Rowan Jessup's LPE for each month from May to September 2026.

*As Riley Quimby* · `/coaching360 Riley Quimby`

**Expected answer:** May 0.70, Jun 0.48, Jul 0.64, Aug 0.35, Sep 0.29. Declining trend.

In [19]:
run("""
SELECT printf('%04d-%02d-01', p.Year, p.Month) AS KpiMonth, p.LPE
FROM org_hierarchy__EDA_Ref_T_Ref_User u
  JOIN branchkpi_sources__BranchKPI_Monthly_EmployeePerformance p ON p.TellerID=u.TellerID
WHERE u.UserDWID=500079 AND p.Year=2026 AND p.Month BETWEEN 5 AND 9
ORDER BY p.Year, p.Month
""")

KpiMonth   | LPE   
-----------+-------
2026-05-01 | 0.7   
2026-06-01 | 0.4762
2026-07-01 | 0.6364
2026-08-01 | 0.35  
2026-09-01 | 0.2857
(5 rows)


### Q13. Help me prepare a KPI coaching conversation with Rowan about LPE.

*As Riley Quimby* · `/coaching360 Riley Quimby`

**Expected answer:** Plan citing LPE 0.29 vs 1.00 (May 0.70 → Jun 0.48 → Jul 0.64 → Aug 0.35 → Sep 0.29), the missed 5-loans-a-week target, calls/contacts short of goal. Also uses Q1 and Q11.

In [20]:
run("""
SELECT printf('%04d-%02d-01', p.Year, p.Month) AS KpiMonth, p.LPE, 1.0 AS LpeGoal, p.TotalLoans, p.DACC_Loans, p.FirstLienPercentage, p.DLQ3060DayPerc_Actual AS Dlq30To89Day
FROM org_hierarchy__EDA_Ref_T_Ref_User u
  JOIN branchkpi_sources__BranchKPI_Monthly_EmployeePerformance p ON p.TellerID=u.TellerID
WHERE u.UserDWID=500079 AND p.Year=2026 AND p.Month BETWEEN 4 AND 9
ORDER BY p.Year, p.Month
""")

KpiMonth   | LPE    | LpeGoal | TotalLoans | DACC_Loans | FirstLienPercentage | Dlq30To89Day
-----------+--------+---------+------------+------------+---------------------+-------------
2026-04-01 | 0.619  | 1.0     | 13         | 3          | 44.02               | 0.039483    
2026-05-01 | 0.7    | 1.0     | 14         | 1          | 42.68               | 0.054662    
2026-06-01 | 0.4762 | 1.0     | 10         | 3          | 33.53               | 0.047558    
2026-07-01 | 0.6364 | 1.0     | 14         | 1          | 43.71               | 0.051106    
2026-08-01 | 0.35   | 1.0     | 7          | 1          | 38.92               | 0.046719    
2026-09-01 | 0.2857 | 1.0     | 6          | 2          | 32.56               | 0.040913    
(6 rows)


### Q14. Draft a recognition note for Selena Driscoll on her September results.

*As Riley Quimby* · `/coaching360 Riley Quimby`

**Expected answer:** Note citing LPE 0.95 (up from 0.80), 4 DACC loans, 30–89 DLQ 2.98% (only one under the 3.40% goal), 54.6% first lien (highest at Laurel Fork).

In [21]:
run("""
SELECT printf('%04d-%02d-01', p.Year, p.Month) AS KpiMonth, p.LPE, p.DACC_Loans, p.DLQCurrentMonthPerc_Actual AS DlqCurrentMonth, p.DLQCurrentMonthPerc_Goal AS DlqCurrentMonthGoal, p.DLQ3060DayPerc_Actual AS Dlq30To89Day, p.DLQ3089DayPerc_Goal AS Dlq30To89DayGoal, p.FirstLienPercentage
FROM org_hierarchy__EDA_Ref_T_Ref_User u
  JOIN branchkpi_sources__BranchKPI_Monthly_EmployeePerformance p ON p.TellerID=u.TellerID
WHERE u.Name='Selena Driscoll' AND p.Year=2026 AND p.Month IN (8, 9)
ORDER BY p.Year DESC, p.Month DESC
""")

KpiMonth   | LPE    | DACC_Loans | DlqCurrentMonth | DlqCurrentMonthGoal | Dlq30To89Day | Dlq30To89DayGoal | FirstLienPercentage
-----------+--------+------------+-----------------+---------------------+--------------+------------------+--------------------
2026-09-01 | 0.9524 | 4          | 0.071149        | 0.08                | 0.029812     | 0.034            | 54.58              
2026-08-01 | 0.8    | 4          | 0.066912        | 0.08                | 0.037503     | 0.034            | 52.36              
(2 rows)


### Q15. Do I have any coaching records waiting for my acknowledgement?

*As Rowan Jessup* · `/coaching360 Rowan Jessup`

**Expected answer:** Yes, 4, all LPE: 01-16, 04-02, 05-11 (KPI) and 07-30 (branch visit). Oldest waiting 9+ months.

In [22]:
run("""
SELECT CoachingDate, TriggerType, KpiFocusCode, CAST(julianday('2026-10-20') - julianday(date(CoachingDate)) AS INTEGER) AS DaysWaiting
FROM coaching360_app_workflow__dbo_CoachingRecords
WHERE EmployeeId=500079 AND Status='PendingAcknowledgement'
ORDER BY CoachingDate
""")

CoachingDate            | TriggerType | KpiFocusCode | DaysWaiting
------------------------+-------------+--------------+------------
2026-01-16 00:00:00.000 | kpi         | lpe          | 277        
2026-04-02 00:00:00.000 | kpi         | lpe          | 201        
2026-05-11 00:00:00.000 | kpi         | lpe          | 162        
2026-07-30 00:00:00.000 | branchVisit | lpe          | 82         
(4 rows)


## Act 4 · Oversight

### Q16. Give me an overview of the branches in my district.

*As Felicia Umberto* · `/coaching360 Felicia Umberto`

**Expected answer:** District 902 Shenandoah, Sept: 8007 0.94 LPE / 4.11% DLQ · 8008 0.80 / 3.22% · 8009 0.99 / 2.62% · 8010 0.95 / 3.78% · 8011 0.64 / 4.19% · 8012 0.86 / 3.15%. Laurel Fork lowest; Stonewall Junction leads.

In [23]:
run("""
SELECT p.Branch, b.BranchName, p.Direct_Loans_MTD_LPE AS LPE, p.DLQ3060DayPerc_Actual AS Dlq30To89Day, p.FirstLienPercentage
FROM branchkpi_sources__BranchKPI_Monthly_BranchPerformance p
  JOIN org_hierarchy__EDA_Ref_T_Ref_Branch b ON b.BranchNumber=p.Branch
  JOIN org_hierarchy__EDA_Ref_T_Ref_District d ON d.DistrictDWID=b.DistrictDWID
WHERE d.DistrictName='Shenandoah' AND p.Year=2026 AND p.Month=9
ORDER BY p.Branch
""")

Branch | BranchName          | LPE    | Dlq30To89Day | FirstLienPercentage
-------+---------------------+--------+--------------+--------------------
8007   | Blue Ridge Crossing | 0.9405 | 0.041068     | 54.06              
8008   | Rappahannock Hills  | 0.7976 | 0.032196     | 47.22              
8009   | Stonewall Junction  | 0.9881 | 0.026218     | 49.14              
8010   | Pine Bluff          | 0.9524 | 0.037827     | 51.44              
8011   | Laurel Fork         | 0.6429 | 0.041896     | 42.19              
8012   | Shenandoah Mills    | 0.8571 | 0.031519     | 48.69              
(6 rows)


### Q17. Which branches have the most coaching records pending acknowledgement or open tasks?

*As Felicia Umberto* · `/coaching360 Felicia Umberto`

**Expected answer:** Pending: Pine Bluff 6, Laurel Fork 5, Blue Ridge Crossing 3, others 2. Open tasks: Shenandoah Mills 36, Pine Bluff 33, Blue Ridge Crossing 25, Laurel Fork 25, Stonewall Junction 24, Rappahannock Hills 21.

In [24]:
run("""
SELECT b.BranchName, SUM((SELECT COUNT(*)
FROM coaching360_app_workflow__dbo_CoachingRecords r
WHERE r.EmployeeId=u.UserDWID AND r.Status='PendingAcknowledgement')) AS PendingAcknowledgements, SUM((SELECT COUNT(*)
FROM coaching360_app_workflow__dbo_CoachingTasks t
WHERE t.AssignedToEmployeeId=u.UserDWID AND t.Status NOT IN ('Completed','Cancelled'))) AS OpenTasks
FROM org_hierarchy__EDA_Ref_T_Ref_User u
  JOIN org_hierarchy__EDA_Ref_T_Ref_Branch b ON b.BranchDWID=u.BranchDWID
  JOIN org_hierarchy__EDA_Ref_T_Ref_District d ON d.DistrictDWID=b.DistrictDWID
WHERE d.DistrictName='Shenandoah'
GROUP BY b.BranchName
ORDER BY PendingAcknowledgements DESC, OpenTasks DESC
""")

BranchName          | PendingAcknowledgements | OpenTasks
--------------------+-------------------------+----------
Pine Bluff          | 6                       | 33       
Laurel Fork         | 5                       | 25       
Blue Ridge Crossing | 3                       | 25       
Shenandoah Mills    | 2                       | 36       
Stonewall Junction  | 2                       | 24       
Rappahannock Hills  | 2                       | 21       
(6 rows)


### Q18. Help me plan my next visit to Laurel Fork. What should I know before I walk in?

*As Felicia Umberto* · `/coaching360 Felicia Umberto`

**Expected answer:** Lowest LPE (0.64) and DLQ (4.19%) in district; 5 pending acks, 25 open tasks; last visits 10-07, 09-08, 08-27. People detail from Q4, plans from Q9.

In [25]:
run("""
SELECT b.BranchName, p.Direct_Loans_MTD_LPE AS LPE, p.DLQ3060DayPerc_Actual AS Dlq30To89Day, RANK() OVER (ORDER BY p.Direct_Loans_MTD_LPE DESC) AS LpeRank, RANK() OVER (ORDER BY p.DLQ3060DayPerc_Actual) AS DlqRank
FROM branchkpi_sources__BranchKPI_Monthly_BranchPerformance p
  JOIN org_hierarchy__EDA_Ref_T_Ref_Branch b ON b.BranchNumber=p.Branch
  JOIN org_hierarchy__EDA_Ref_T_Ref_District d ON d.DistrictDWID=b.DistrictDWID
WHERE d.DistrictName='Shenandoah' AND p.Year=2026 AND p.Month=9
ORDER BY b.BranchName='Laurel Fork' DESC, LpeRank
""")

BranchName          | LPE    | Dlq30To89Day | LpeRank | DlqRank
--------------------+--------+--------------+---------+--------
Laurel Fork         | 0.6429 | 0.041896     | 6       | 6      
Stonewall Junction  | 0.9881 | 0.026218     | 1       | 1      
Pine Bluff          | 0.9524 | 0.037827     | 2       | 4      
Blue Ridge Crossing | 0.9405 | 0.041068     | 3       | 5      
Shenandoah Mills    | 0.8571 | 0.031519     | 4       | 2      
Rappahannock Hills  | 0.7976 | 0.032196     | 5       | 3      
(6 rows)


### Q19. What follow-ups came out of my October 7 visit to Laurel Fork, and are they done?

*As Felicia Umberto* · `/coaching360 Felicia Umberto`

**Expected answer:** One: Collections, "Work the 30-day bucket every morning and review the aging report Fridays," owner Riley Quimby, due 10-21. Completed 10-19, two days early.

In [27]:
run("""
SELECT v.VisitDate, tp.TopicTitle, tp.ActionPlan, tp.AssignedOwnerName, t.DueDate AS TaskDueDate, t.Status AS FollowUpStatus, t.CompletedUtc AS TaskCompletedDate
FROM coaching360_app_workflow__dbo_BranchVisits v
  JOIN org_hierarchy__EDA_Ref_T_Ref_Branch b ON b.BranchDWID=v.OrganizationalUnitId
  JOIN coaching360_app_workflow__dbo_BranchVisitTopics tp ON tp.BranchVisitId=v.BranchVisitId
  LEFT JOIN coaching360_app_workflow__dbo_CoachingTasks t ON t.ParentType='BranchVisit' AND t.ParentId=v.BranchVisitId
WHERE b.BranchName='Laurel Fork' AND date(v.VisitDate)='2026-10-07' AND tp.FollowUpRequired=1
""")

VisitDate               | TopicTitle  | ActionPlan                                                   | AssignedOwnerName | TaskDueDate             | FollowUpStatus | TaskCompletedDate      
------------------------+-------------+--------------------------------------------------------------+-------------------+-------------------------+----------------+------------------------
2026-10-07 00:00:00.000 | Collections | Work the 30-day bucket every morning and review the aging re | Riley Quimby      | 2026-10-21 00:00:00.000 | Completed      | 2026-10-19 15:28:54.751
(1 rows)


### Q20. How is District 902 doing compared to the other districts in my region?

*As Keisha Foxworth* · `/coaching360 Keisha Foxworth`

**Expected answer:** 901 Chesapeake 0.96 LPE, DLQ score 108.8, ROA 100.0 · 902 Shenandoah 0.87, 96.8, 90.1 · 903 Susquehanna 0.90, 109.7, 110.9. DLQ score is goal/actual; above 100 beats goal.

In [28]:
run("""
SELECT d.DistrictName, p.DistrictLPE, p.DLQ_30_89_Percentage AS DlqScore, p.ROA_Percentage AS RoaScore, p.FirstLienPercentage
FROM branchkpi_sources__BranchKPI_Monthly_DistrictPerformance p
  JOIN org_hierarchy__EDA_Ref_T_Ref_District d ON CAST(d.DistrictNumber AS INTEGER)=p.DistrictNumber
WHERE p.Year=2026 AND p.Month=9 AND p.DistrictNumber IN (901,902,903)
ORDER BY p.DistrictNumber
""")

DistrictName | DistrictLPE | DlqScore | RoaScore | FirstLienPercentage
-------------+-------------+----------+----------+--------------------
Chesapeake   | 0.9615      | 108.75   | 100.0    | 46.0               
Shenandoah   | 0.87        | 96.81    | 90.14    | 49.14              
Susquehanna  | 0.9014      | 109.67   | 110.92   | 45.53              
(3 rows)


### Q21. Are my district managers keeping up with branch visits?

*As Keisha Foxworth* · `/coaching360 Keisha Foxworth`

**Expected answer:** As of 2026-10-20, completed / scheduled / overdue: Felicia Umberto 76 / 2 / 1; Nolan Ingram 74 / 4 / 0; Olivia Northcott 76 / 5 / 1. October so far: Felicia 7, Olivia 5, Nolan 2.

In [29]:
run("""
SELECT d.DMName, (SELECT COUNT(*)
FROM coaching360_app_workflow__dbo_BranchVisits v
  JOIN org_hierarchy__EDA_Ref_T_Ref_Branch b ON b.BranchDWID=v.OrganizationalUnitId
WHERE b.DistrictDWID=d.DistrictDWID AND v.VisitStatus='Submitted' AND v.IsArchived=0) AS completed, (SELECT COUNT(*)
FROM coaching360_app_workflow__dbo_BranchVisitPlanningPlaceholders x
  JOIN org_hierarchy__EDA_Ref_T_Ref_Branch b ON b.BranchDWID=x.OrganizationalUnitId
WHERE b.DistrictDWID=d.DistrictDWID AND x.Status='Active' AND date(x.VisitDate)>'2026-10-20') AS scheduled, (SELECT COUNT(*)
FROM coaching360_app_workflow__dbo_BranchVisitPlanningPlaceholders x
  JOIN org_hierarchy__EDA_Ref_T_Ref_Branch b ON b.BranchDWID=x.OrganizationalUnitId
WHERE b.DistrictDWID=d.DistrictDWID AND x.Status='Active' AND date(x.VisitDate)<='2026-10-20') AS overdue, (SELECT COUNT(*)
FROM coaching360_app_workflow__dbo_BranchVisits v
  JOIN org_hierarchy__EDA_Ref_T_Ref_Branch b ON b.BranchDWID=v.OrganizationalUnitId
WHERE b.DistrictDWID=d.DistrictDWID AND v.IsArchived=0 AND v.VisitDate>='2026-10-01') AS october
FROM org_hierarchy__EDA_Ref_T_Ref_District d
  JOIN org_hierarchy__EDA_Ref_T_Ref_Region r ON r.RegionDWID=d.RegionDWID
WHERE r.VPName='Keisha Foxworth'
ORDER BY d.DMName
""")

DMName           | completed | scheduled | overdue | october
-----------------+-----------+-----------+---------+--------
Felicia Umberto  | 76        | 2         | 1       | 7      
Nolan Ingram     | 74        | 4         | 0       | 2      
Olivia Northcott | 76        | 5         | 1       | 5      
(3 rows)
